<h1>
    Previsão inadimplência com ML
</h1>
<hr>

<p>
    Esse projeto utiliza o dataset
    <a href="https://www.kaggle.com/datasets/uciml/default-of-credit-card-clients-dataset/data"><b>Default of Credit Card Clients</b></a>
    fornecido pela
    <a href="https://archive.ics.uci.edu/"><b>UCI</b></a> e destribuido pela
    <a href="https://www.kaggle.com/"><b>Kaggle</b></a>
</p>

<h2>Objetivo</h2>

<p>
    O objetivo deste projeto é treinar um modelo de classificação binária com os dados do dataset para identificar se um cliente do banco irá ou não entrar em inadimplência no
    próximo mês. Os dados contém 25 preditores e 30.000 observações onde 80% (24.000) dos dados  serão usados para o treinamento e validação, enquanto os 20% restantes (6.000)
    serão usados para testar o modelo.
</p>

<p>
     Após treinado, o modelo gerará apenas dois tipos de previsões:
    <br>
    <b>0</b> - Indica que o cliente permanecerá adimplente.
    <br>
    <b>1</b> - Indica que o cliente entrará em inadimplência.
</p>

<h2>Bibliotecas necessárias</h2>

<ul>
    <li>
        Pandas - será usado para manipular e transformar os dados brutos
    </li>
    <li>
        scikit-learn / train_test_split - será usado para dividir o dataset em treinamento e validação
    </li>
    <li>
        scikit-learn / RandomForestClassifier - será usado para o treinamento do modelo
    </li>
</ul>
<hr>
<h3>Instalando e importando bibliotecas</h3>

<code>pip install pandas scikit-learn matplotlib jupyterlab imbalanced-learn</code>

In [2]:
import matplotlib.pyplot as plt
import pandas as pd

from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn import metrics

<h2>
    <b>Carregando o dataset</b>
</h2>

In [2]:
df = pd.read_csv('../data/raw/default_of_credit_card_clients.csv')

<h2>
    <b>Visão geral dos dados</b>
</h2>

<h3>
    Quantidade total de linhas e colunas
</h3>

In [3]:
df.shape

(30000, 25)

<h3>
    Total de elementos
</h3>

In [4]:
df.size

750000

<h3>
    Listando todas as colunas
</h3>

In [5]:
df.columns.tolist()

['ID',
 'LIMIT_BAL',
 'SEX',
 'EDUCATION',
 'MARRIAGE',
 'AGE',
 'PAY_0',
 'PAY_2',
 'PAY_3',
 'PAY_4',
 'PAY_5',
 'PAY_6',
 'BILL_AMT1',
 'BILL_AMT2',
 'BILL_AMT3',
 'BILL_AMT4',
 'BILL_AMT5',
 'BILL_AMT6',
 'PAY_AMT1',
 'PAY_AMT2',
 'PAY_AMT3',
 'PAY_AMT4',
 'PAY_AMT5',
 'PAY_AMT6',
 'default.payment.next.month']

<h3>
    Dicionário das colunas
</h3>

<p>
    <b></b>
</p>
<p>
    <b>ID</b> - Identidade unica de cada cliente
</p>
<p>
    <b>LIMIT_BAL</b> - Limite do Saldo de Crédito
</p>
<p>
    <b>SEX</b> - Sexo do cliente (1 = male, 2 = female)
</p>
<p>
    <b>EDUCATION</b> - Nível de educação (1 = graduate school, 2 = university, 3 = high school, 4 = others, 5 = unknown, 6 = unknown)
</p>
<p>
    <b>MARRIAGE</b> - Estado civil (1 = married, 2 = single, 3 = others)
</p>
<p>
    <b>AGE</b> - Idade em anos
</p>
<p>
    <b>PAY_0 até PAY_6</b> - Remete ao status de pagamentos dos meses setembro (PAY_0) até abril (PAY_6) *Mais detalhes embaixo*
</p>
<p>
    <b>BILL_AMT1 até BILL_AMT6</b> - Remetem aos valores das faturas de setembro (BILL_AMT1) até abril (BILL_AMT6)
</p>
<p> 
    <b>PAY_AMT1 até PAY_AMT6</b> - Remete ao valor dos pagamentos das faturas de setembro (PAY_AMT1) até abril (PAY_AMT6)
</p>
<p>
    <b>default.payment.next.month</b> - Indica se o cliente entrará em inadimplência no próximo mês (1=yes, 0=no)
</p>

<hr>

<p>
    Mais detalhes:
    <br>
    As colunas PAY_n não começam a partir do 1, e sim do 0 e depois pula para o 2 e segue normalmente até o 6. Isso será corrigido no pré-processamento.
    <br>
    Valores que <b>PAY_0, PAY_2 ... PAY_6</b> podem receber:
    <ul>
        <li>
            -1 = pay duly (Pagamento realizado corretamente)
        </li>
        <li>
            1 to 8 = Months of payment delay (Pagamento com atraso de 1 a 8 meses)
        </li>
        <li>
            9 = Payment delay for nine months and above. (Pagamento com atraso de 9 mese ou mais)
        </li>
    </ul>
</p>
    
    
    
    
    
    
    


<h3>
    As primeiras linhas
</h3>

In [6]:
df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


<h3>
    As ultimas linhas
</h3>

In [7]:
df.tail()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
29995,29996,220000.0,1,3,1,39,0,0,0,0,...,88004.0,31237.0,15980.0,8500.0,20000.0,5003.0,3047.0,5000.0,1000.0,0
29996,29997,150000.0,1,3,2,43,-1,-1,-1,-1,...,8979.0,5190.0,0.0,1837.0,3526.0,8998.0,129.0,0.0,0.0,0
29997,29998,30000.0,1,2,2,37,4,3,2,-1,...,20878.0,20582.0,19357.0,0.0,0.0,22000.0,4200.0,2000.0,3100.0,1
29998,29999,80000.0,1,3,1,41,1,-1,0,0,...,52774.0,11855.0,48944.0,85900.0,3409.0,1178.0,1926.0,52964.0,1804.0,1
29999,30000,50000.0,1,2,1,46,0,0,0,0,...,36535.0,32428.0,15313.0,2078.0,1800.0,1430.0,1000.0,1000.0,1000.0,1


<h3>
    Resumo estatístico do conjunto de dados
</h3>

In [8]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
ID,30000.0,15000.500000,8660.398374,1.0,7500.75,15000.5,22500.25,30000.0
LIMIT_BAL,30000.0,167484.322667,129747.661567,10000.0,50000.00,140000.0,240000.00,1000000.0
SEX,30000.0,1.603733,0.489129,1.0,1.00,2.0,2.00,2.0
EDUCATION,30000.0,1.853133,0.790349,0.0,1.00,2.0,2.00,6.0
MARRIAGE,30000.0,1.551867,0.521970,0.0,1.00,2.0,2.00,3.0
AGE,30000.0,35.485500,9.217904,21.0,28.00,34.0,41.00,79.0
PAY_0,30000.0,-0.016700,1.123802,-2.0,-1.00,0.0,0.00,8.0
PAY_2,30000.0,-0.133767,1.197186,-2.0,-1.00,0.0,0.00,8.0
PAY_3,30000.0,-0.166200,1.196868,-2.0,-1.00,0.0,0.00,8.0
PAY_4,30000.0,-0.220667,1.169139,-2.0,-1.00,0.0,0.00,8.0


<h3>
    Tipo dos dados das colunas
</h3>

In [9]:
df.dtypes

ID                              int64
LIMIT_BAL                     float64
SEX                             int64
EDUCATION                       int64
MARRIAGE                        int64
AGE                             int64
PAY_0                           int64
PAY_2                           int64
PAY_3                           int64
PAY_4                           int64
PAY_5                           int64
PAY_6                           int64
BILL_AMT1                     float64
BILL_AMT2                     float64
BILL_AMT3                     float64
BILL_AMT4                     float64
BILL_AMT5                     float64
BILL_AMT6                     float64
PAY_AMT1                      float64
PAY_AMT2                      float64
PAY_AMT3                      float64
PAY_AMT4                      float64
PAY_AMT5                      float64
PAY_AMT6                      float64
default.payment.next.month      int64
dtype: object

<h3>
    Verificando se há valores duplicados // passsivel a ser deletado
</h3>

In [10]:
df.duplicated().sum()

np.int64(0)

<h3>
    Verificando se há colunas sem valores declarados // passsivel a ser deletado
</h3>

In [11]:
df.isnull().sum()

ID                            0
LIMIT_BAL                     0
SEX                           0
EDUCATION                     0
MARRIAGE                      0
AGE                           0
PAY_0                         0
PAY_2                         0
PAY_3                         0
PAY_4                         0
PAY_5                         0
PAY_6                         0
BILL_AMT1                     0
BILL_AMT2                     0
BILL_AMT3                     0
BILL_AMT4                     0
BILL_AMT5                     0
BILL_AMT6                     0
PAY_AMT1                      0
PAY_AMT2                      0
PAY_AMT3                      0
PAY_AMT4                      0
PAY_AMT5                      0
PAY_AMT6                      0
default.payment.next.month    0
dtype: int64

<h2>
    <b>
        Análise da variável alvo  // default.payment.next.month
    </b>
</h2>

In [12]:
# Criando método para gerar e salvar o gráfico da variavel alvo .

def gerar_grafico_variavel_alvo(var_alvo, nome_arquivo):
    fig, ax = plt.subplots(figsize=(5,5))
    valores = var_alvo.value_counts()
    barras = ax.bar(valores.index, valores.values)
    
    ax.bar_label(barras, padding=2 , fontsize=9, fontweight='bold')
    ax.set_title('Distribuição de Inadimplência no Próximo Mês', fontsize=14, pad=15)
    ax.set_xlabel('Situação', fontsize=10, labelpad=10)
    ax.set(
        xticks = [1,0],
        xticklabels=['Inadimplentes [1]', 'Adimplentes [0]']
    )
    
    plt.savefig(f'../imagens/{nome_arquivo}.png', dpi = 400)
    
    plt.show()

In [ ]:
# Chamando o método com dataframe bruto.
gerar_grafico_variavel_alvo(df['default.payment.next.month'], 'dados_brutos-var_dependente_discrepancia')

<p><strong>Output pré-renderizado:</strong></p>
<div style="text-align: center;">
    <img src='../imagens/dados_brutos-var_dependente_discrepancia.png' width="500" alt='Gráfico mostrando a discrepancia entre os valores da coluna default.payment.next.month'>
</div>

<p>
    Como observado, a variável alvo apresenta uma distribuição desbalanceada, onde os clientes inadimplentes representam cerca de 22% da base total e os adimplentes 78%. Em alguns modelos de Machine Learning, essa distribuição pode fazer com que eles não identifiquem padrões importantes da classe minoritária e criem um viés em favor da classe majoritária, tornando-se assim um modelo tendencioso.
</p>

<p>
    Para lidar com esse cenário, utilizaremos o modelo <code>RandomForestClassifier</code> em conjunto com a técnica de sobreamostragem SMOTE. O modelo funciona criando múltiplas árvores de decisão, cada uma é gerada a partir de uma amostra bootstrap e, em cada nó, a divisão é definida por um subconjunto aleatório de características, garantindo que as árvores sejam independentes, diversificadas e com um baixo viés. Já o SMOTE, atuará criando exemplos sintéticos dos clientes inadimplentes, equilibrando a base de treino e permitindo que o modelo aprenda de forma mais eficaz as características associadas à inadimplência, reduzindo o viés para a classe majoritária.
</p>

<p>
    Após o treinamento, o resultado final para problemas de classificação é obtido através do sistema de votação majoritária: o modelo coleta as previsões de todas as árvores individuais e define a classe mais frequente como a resposta final.
</p>

<h2>
    <b>Pre-processamento</b>
</h2>

<p>
    Observando o dataset com detalhes fica claro a presença de valores redundantes e até mesmo valores sem nenhum significado declarado.
    <br>
    O objetivo dessa seção é limpar esses valores para que os dados fiquem mais objetivos e consequentemente aumentem a acurácia do modelo. 
</p>

<h3>
    Coluna ID
</h3>
<p>
    Como essa coluna não é um indicador preditivo, será excluida do dataset.
</p>

df.drop(columns='ID', inplace=True)

In [14]:
df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


<h3>
    Coluna EDUCATION
</h3>
<p>Valores aceitos: 1 = graduate school, 2 = university, 3 = high school, 4 = others, 5 = unknown, 6 = unknown</p>

In [15]:
df['EDUCATION'].value_counts()

EDUCATION
2    14030
1    10585
3     4917
5      280
4      123
6       51
0       14
Name: count, dtype: int64

<p>
    O resultado apresenta a contagem de ocorrências para cada categoria de escolaridade.
</p>

<p>
    Ao analisar tanto os valores presentes na coluna quanto os valores aceitos declarados na documentação, fica evidente que as categorias 0, 5, 6 podem ser agrupadas na categoria 4 (others). Já que as categorias 5 e 6 são redundantes (unknown) e a categoria 0 além de não ter sida declarada na documentação, tem baixa representatividade e se encaixa na categoria genérica.
</p>
<p> 
    Para alterar as categorias usamos <code>df['EDUCATION'].replace([0, 5, 6], 4)</code>, o método <code>.replace()</code> irá iterar sobre cada valor na coluna EDUCATION, substituirá
    os valores 0, 5, 6 para o valor 4 e retornará uma nova coluna EDUCATION.
</p>

In [16]:
df['EDUCATION'] = df['EDUCATION'].replace([0, 5, 6], 4)
df['EDUCATION'].value_counts()

EDUCATION
2    14030
1    10585
3     4917
4      468
Name: count, dtype: int64

<p>
    Com essas alterações feitas, podemos aplicar o One-Hot Enconding.
</p>

<p>
    O conceito de One-Hot Encoding visa transformar categorias de uma tabela como <code>1 = graduate school, 2 = university, 3 = high school, 4 = others</code>, da tabela EDUCATION, em colunas separadas onde os únicos valores são <code>1</code> que indica a presença da categoria ou <code>0</code> que indica a ausência.
</p>

In [17]:
df['POS_GRADUACAO'] = (df['EDUCATION'] == 1).astype(int)
df['UNIVERSIDADE'] = (df['EDUCATION'] == 2).astype(int)
df['ENSINO_MEDIO'] = (df['EDUCATION'] == 3).astype(int)
df['OUTROS'] = (df['EDUCATION'] == 4).astype(int)

df.drop('EDUCATION', axis = 1, inplace = True) 

In [18]:
df.head()

,ID,LIMIT_BAL,SEX,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month,POS_GRADUACAO,UNIVERSIDADE,ENSINO_MEDIO,OUTROS
0,1,20000.0,2,1,24,2,2,-1,-1,-2,...,689.0,0.0,0.0,0.0,0.0,1,0,1,0,0
1,2,120000.0,2,2,26,-1,2,0,0,0,...,1000.0,1000.0,1000.0,0.0,2000.0,1,0,1,0,0
2,3,90000.0,2,2,34,0,0,0,0,0,...,1500.0,1000.0,1000.0,1000.0,5000.0,0,0,1,0,0
3,4,50000.0,2,1,37,0,0,0,0,0,...,2019.0,1200.0,1100.0,1069.0,1000.0,0,0,1,0,0
4,5,50000.0,1,1,57,-1,0,-1,0,0,...,36681.0,10000.0,9000.0,689.0,679.0,0,0,1,0,0


<h3>Coluna MARRIAGE</h3>
<p>
    Valores aceitos: (1=married, 2=single, 3=others)
</p>

In [19]:
df['MARRIAGE'].value_counts()

MARRIAGE
2    15964
1    13659
3      323
0       54
Name: count, dtype: int64

<p>
    O resultado apresenta a contagem de ocorrências para cada categoria de estatus civil.
</p>
<p>
    Ao analisar tanto o resultado quanto os valores aceitos, fica evidente que a categoria 0 podm ser agrupada na categoria 3 (others). Já que a categoria 0 além de não ter sida
    declarada na documentação, tem baixa representatividade e se encaixa na categoria genérica.
</p>
<p>
    Para alterar as categorias usamos <code>df['MARRIAGE'].replace([0], 3)</code>, o método <code>.replace()</code> irá iterar sobre cada valor na coluna MARRIAGE,
    substituirá o valor 0 para o valor 3 e retornará uma nova coluna MARRIAGE.
</p>

In [20]:
df['MARRIAGE'] = df['MARRIAGE'].replace([0], 3)
df['MARRIAGE'].value_counts()

MARRIAGE
2    15964
1    13659
3      377
Name: count, dtype: int64

<p>
    Com essas alterações feitas, podemos aplicar o One-Hot Enconding.
</p>

In [21]:
df['CASADO'] = (df['MARRIAGE'] == 1).astype(int)
df['SOLTEIRO'] = (df['MARRIAGE'] == 2).astype(int)

df.drop('MARRIAGE', axis = 1, inplace = True) 

In [22]:
df.head()

,ID,LIMIT_BAL,SEX,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,...,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month,POS_GRADUACAO,UNIVERSIDADE,ENSINO_MEDIO,OUTROS,CASADO,SOLTEIRO
0,1,20000.0,2,24,2,2,-1,-1,-2,-2,...,0.0,0.0,0.0,1,0,1,0,0,1,0
1,2,120000.0,2,26,-1,2,0,0,0,2,...,1000.0,0.0,2000.0,1,0,1,0,0,0,1
2,3,90000.0,2,34,0,0,0,0,0,0,...,1000.0,1000.0,5000.0,0,0,1,0,0,0,1
3,4,50000.0,2,37,0,0,0,0,0,0,...,1100.0,1069.0,1000.0,0,0,1,0,0,1,0
4,5,50000.0,1,57,-1,0,-1,0,0,0,...,9000.0,689.0,679.0,0,0,1,0,0,1,0


<h2>
    <b>Dividindo os dados</b>
</h2>

<p>
    A divisão dos dados é essencial em Machine Learning pois permite treinar e validar o modelo em um conjunto de dados e avaliá-lo em outro que ele nunca viu. Isso ajuda a
    verificar se o modelo realmente aprendeu padrões generalizáveis ou se apenas memorizou os dados de treino, evitando problemas como overfitting.
</p>
<p>
    Como mencionado anteriormente, o dataset será dividido em 80% para treinamento e validação e 20% para teste do modelo. Para isso, utilizaremos o método
    <code>train_test_split</code>, que
    separará os dados em dois conjuntos: <code>train_data</code> e <code>test_data</code>. Posteriormente, esses conjuntos serão utilizados para criar as variáveis independentes
    <code>X_train</code> e <code>X_test</code> e variáveis independentes do modelo <code>y_train</code> e <code>y_test</code>.
</p>

In [23]:
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42, stratify=df['default.payment.next.month'])

<p>
    <ul>
        <li>
            <code>df</code> : O objeto dataframe que faz referência ao csv carregado no inicio desse notebook.
        </li>
        <li>
            <code>test_size=0.2</code> : Nesse parâmetro é definido a porcentagem que será atribuída ao conjunto de validação.
        </li>
        <li>
            <code>random_state=42</code> : Esse parâmetro define um número que normalmente é gerado de forma aleatória que determina como os dados serão embaralhados antes de serem divididos. Por conta dessa definição explícita, torna a divisão dos dados reproduzível.  
        </li>
        <li>
            <code>stratify=df['default.payment.next.month']</code> : O parâmetro <code>stratify</code> garante que a proporção da variável alvo seja preservada tanto no conjunto de treino quanto no de teste, evitando que a divisão aleatória crie conjuntos desbalanceados.
        </li>
    </ul>
</p>

<h4>
    Quantidade total de linhas e colunas em <code>train_data</code>
</h4>

In [24]:
train_data.shape

(24000, 29)

<h4>
    Variável alvo em <code>train_data</code>
</h4>

In [ ]:
gerar_grafico_variavel_alvo(train_data['default.payment.next.month'], 'train_data-var_dependente_discrepancia')

<p><strong>Output pré-renderizado:</strong></p>
<div style="text-align: center;">
    <img src='../imagens/train_data-var_dependente_discrepancia.png' width="500" alt='Gráfico mostrando a discrepancia entre os valores da coluna default.payment.next.month da tabela train_data'>
</div>

<h4>
    Quantidade total de linhas e colunas em <code>test_data</code>
</h4>

In [26]:
test_data.shape

(6000, 29)

<h4>
    Variável alvo em <code>test_data</code>
</h4>

In [ ]:
gerar_grafico_variavel_alvo(test_data['default.payment.next.month'], 'test_data-var_dependente_discrepancia') 

<p><strong>Output pré-renderizado:</strong></p>
<div style="text-align: center;">
    <img src='../imagens/test_data-var_dependente_discrepancia.png' width="500" alt='Gráfico mostrando a discrepancia entre os valores da coluna default.payment.next.month da tabela test_data'>
</div>

<h4>
    Persistindo <code>train_data</code> e <code>test_data</code> em arquivos CSV.
</h4>

<p>
    Com os dados persistidos, temos agora os conjuntos de dados que serão usados aqui e que poderão ser usados em outros projetos de machine learning.
</p>

In [28]:
train_data.to_csv("../data/processed/train_data.csv", index=False)

In [29]:
test_data.to_csv("../data/processed/test_data.csv", index=False)

<h3>
    Criando as variaveis dependentes e independentes.
</h3>

In [30]:
X_train = train_data.drop(columns = ['default.payment.next.month'])
y_train = train_data['default.payment.next.month']

In [31]:
X_test = test_data.drop(columns = ['default.payment.next.month'])
y_test = test_data['default.payment.next.month']

<h2>
    <b>Aplicando SMOTE na variável alvo desbalanceada</b>
</h2>

<h1>para alterar</h1>

In [32]:
smote = SMOTE(sampling_strategy = 'minority', random_state=19)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

In [ ]:
gerar_grafico_variavel_alvo(y_train_res, "y_train_resampled--var_dependente_discrepancia")

<p>
<strong>Output pré-renderizado:</strong>
</p>

<div style="text-align: center;">
    <img src='../imagens/y_train_resampled--var_dependente_discrepancia.png' width="500" alt='Gráfico mostrando os valores da variável dependente após a aplicação do SMOTE'>
</div>

<h2>
    <b>Validação cruzada usando <code>RandomizedSearchCV</code></b>
</h2>

<p>
    A validação serve para ajustar os hiperparâmetros do modelo, selecionando a configuração que garante a melhor performance preditiva do modelo e capacidade de generalização em dados não vistos.
</p>

<p>
    O "CV" no nome da classe <code>RandomizedSearchCV</code> se refere ao cross-validation, ou validação cruzada. Nesse processo, os dados passados ao método
    <code>.fit()</code> são divididos em uma quantidade <code>n</code> de subconjuntos (folds), nos quais parte é utilizada para treinar o modelo e outra
    para validação. Esses subconjuntos são utilizados de forma alternada ao longo de <code>n</code> rodadas, permitindo avaliar o desempenho médio do modelo e selecionar
    os hiperparâmetros que produzem os melhores resultados.
</p>

<p>
    A classe <code>RandomizedSearchCV</code>, da biblioteca scikit-learn, permite realizar múltiplos treinamentos testando diferentes combinações de hiperparâmetros, buscando
    encontrar o modelo com melhor desempenho.
</p>

<h3>
    Dicionário de hiperparâmetros
</h3>

In [34]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None],
    "class_weight": [None, "balanced"]
}

<p>
    A função do <code>param_grid</code> é armazenar, em um dicionário, os nomes dos parâmetros do modelo juntamente com uma sequência de valores que cada parâmetro pode assumir.
    <br>
    É a partir desse dicionário que o <code>RandomizedSearchCV</code> irá gerar diferentes combinações de hiperparâmetros para treinar e validar o modelo <code>RandomForestClassifier</code>.
</p>

<p>
    Significado dos hiperparâmetros:
    <ul>
        <li><code>n_estimators</code> define o número de árvores que serão geradas na floresta.</li>
        <li><code>max_depth</code> define a profundidade que cada árvore terá do nó raiz até o nó folha.</li>
        <li><code>min_samples_split</code> define o mínimo de amostras necessárias para que um nó seja dividido. Se um nó tiver menos amostras que isso, ele não será dividido.</li>
        <li><code>min_samples_leaf</code> define o número mínimo de amostras que devem estar presentes no nó folha após a divisão de um nó.</li>
        <li><code>max_features</code> define o número de características consideradas a cada divisão de um nó na árvore.</li>
        <li><code>class_weight</code> define os pesos atribuídos às classes da variável alvo, podendo dar mais importância a classes menos frequentes.</li>
    </ul>
</p>

<h3>
    Inicializando o modelo Random Forest
</h3>

In [35]:
random_forest_modelo = RandomForestClassifier()

<h3>
    Configurando Random SearchCV
</h3>

In [36]:
random_search = RandomizedSearchCV(
    estimator = random_forest_modelo,
    param_distributions = param_grid,
    n_iter = 100,
    cv = 5,
    n_jobs = -1,
    random_state=19
)

<p>
    Parâmetros:
    <ul>
        <li><code>estimator</code> define o modelo base que queremos otimizar. No nosso caso, é a instância da Random Forest criada anteriormente.</li>
        <li><code>param_distributions</code> é o dicionário que contém os parâmetros que queremos testar e os respectivos valores ou intervalos.</li>
        <li><code>n_iter</code> define quantas combinações aleatórias de parâmetros o algoritmo vai testar.</li>
        <li><code>cv</code> define em quantos subconjuntos (folds) os dados serão divididos para a validação cruzada.</li>
        <li><code>n_jobs</code> define quantos núcleos do processador serão utilizados para executar os testes em paralelo.</li>
        <li><code>random_state</code> garante que os resultados sejam replicáveis.</li>
    </ul>
</p>

<h3>
    Inicializando a busca pelos melhores hiperparâmetros
</h3>

In [37]:
random_search.fit(X_train_res, y_train_res)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestClassifier()
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'class_weight': [None, 'balanced'], 'max_depth': [None, 10, ...], 'max_features': ['sqrt', 'log2', ...], 'min_samples_leaf': [1, 2, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",100
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used her

<h3>
    Melhores Hyperparametros encontrados
</h3>

In [38]:
random_search.best_params_

{'n_estimators': 300,
 'min_samples_split': 2,
 'min_samples_leaf': 1,
 'max_features': 'sqrt',
 'max_depth': None,
 'class_weight': 'balanced'}

<h3>
    Gerando predições com o modelo
</h3>

In [39]:
y_pred = random_search.predict(X_test)

<h2>Métricas do modelo</h2>

<ul>
    <li>
        <h3><strong>Matriz de confusão</strong></h3>
    </li>
</ul>

<p>
    A matriz de confusão é uma ferramenta utilizada para visualizar o desempenho de um modelo de classificação, comparando as predições do modelo com os valores reais.
</p>

<p>
    Como o nosso modelo de machine learning realiza uma classificação binária, os resultados podem ser organizados em quatro categorias:
</p>

<ul>
    <li><code>True positive (TP)</code> acontece quando um cliente inadimplente é corretamente classificado como inadimplente.</li>
    <li><code>True negative (TN)</code> acontece quando um cliente adimplente é corretamente classificado como adimplente.</li>
    <li><code>False positive (FP)</code> acontece quando um cliente adimplente é incorretamente classificado como inadimplente.</li>
    <li><code>False negative (FN)</code> acontece quando um cliente inadimplente incorretamente classificado como adimplente.</li>
</ul>

<p>
    <strong>Mapeamento de Categorias na Matriz de Confusão</strong>
    <br>
    Positive = cliente inadimplente (classe 1)
    <br>
    Negative = cliente adimplente (classe 0)
</p>

<table border="1" style="text-align: center; border-collapse: collapse; width: 400px; font-family: sans-serif;">
  <tr>
    <th colspan="2" rowspan="2"></th>
    <th colspan="2" style="padding: 8px;">Classe Prevista</th>
  </tr>
    
  <tr style="background-color: #f2f2f2;">
    <th style="padding: 8px;">Positive (1)</th>
    <th style="padding: 8px;">Negative (0)</th>
  </tr>
  
  <tr>
    <th rowspan="2">Classe Real</th>
    <th style="background-color: #f2f2f2; padding: 8px;">Positive (1)</th>
    <td style="color: #2e7d32; font-weight: bold; padding: 12px;">True Positive (TP)</td>
    <td style="padding: 12px;">False Negative (FN)</td>
  </tr>
  
  <tr>
    <th style="background-color: #f2f2f2; padding: 8px;">Negative (0)</th>
    <td style="padding: 12px;">False Positive (FP)</td>
    <td style="color: #2e7d32; font-weight: bold; padding: 12px;">True Negative (TN)</td>
  </tr>
</table>

In [ ]:
matriz_confusao = metrics.confusion_matrix(y_test, y_pred, labels=[1, 0])
disp = metrics.ConfusionMatrixDisplay(
    confusion_matrix = matriz_confusao,
    display_labels=['Inadimplente', 'Adimplente'],
    
)
disp.plot(cmap=plt.cm.Blues)

plt.savefig('../imagens/matriz_de_confusao-do-resultado.png', dpi = 400)

plt.show()

<p>
<strong>Output pré-renderizado:</strong>
</p>

<div style="text-align: center;">
    <img src='../imagens/matriz_de_confusao-do-resultado.png' width="600" alt='Matriz de confusão criada usando os dados da ultima predição do modelo'>
</div>

<p>
    Essa matriz de confusão permite observar como o modelo distribui seus acertos e erros entre as classes, mostrando seu comportamento na identificação de inadimplência.
</p>

<ul>
    <li>O modelo classificou corretamente 588 clientes inadimplentes (true positive).</li>
    <li>O modelo classificou incorretamente 739 clientes inadimplentes como adimplentes (false negative).</li>
    <li>O modelo classificou incorretamente 464 clientes adimplentes como inadimplentes (false positive).</li>
    <li>O modelo classificou corretamente 4.209 clientes adimplentes (true negative).</li>
</ul>

<p>
    Isso deixa claro que o modelo apresenta melhor desempenho para a classe adimplente e maior dificuldade na identificação da classe inadimplente, sugerindo um viés em favor dos não inadimplentes.
</p>

---

<ul>
    <li>
        <h3><strong>Accuracy</strong></h3>
    </li>
</ul>

<p>
    Accuracy, ou em PT-BR, acurácia, é uma métrica que mede a frequência em que um modelo de machine learning prevê corretamente o resultado. Essa métrica pode ser obtida dividindo o total de acertos (true positives, true negatives) com o total de previsões.
</p>

<p>
    <b>Definição básica:</b>
</p>

$$
\text{Acurácia} = \frac{\text{total de acertos}}{\text{total de previsões}}
$$

<p>
    <b>Baseada na matriz de confusão:</b>
</p>

$$
\text{Acurácia} = \frac{TP + TN}{TP + TN + FP + FN}
$$

<p>
    No <code>scikit-learn</code>, a função <code>accuracy_score()</code> é utilizada para calcular a acurácia do modelo. O valor retornado está na escala de <code>0</code> a <code>1</code>, onde <code>1</code> corresponde a <code>100%</code> de acurácia.
</p>

In [41]:
acuracia = metrics.accuracy_score(y_test, y_pred)
print("Accuracy: " + str(acuracia))

Accuracy: 0.7995


<p>
    Como pode ser observado, o modelo apresenta uma acurácia de 0,79 (ou 79%). A primeira vista, esse valor pode ser considerado satisfatório, indicando que a maioria das previsões foi realizada corretamente. No entanto, ao analisar os resultados em conjunto com a matriz de confusão, nota-se que o modelo ainda apresenta dificuldades em identificar corretamente os casos de inadimplência, evidenciado pelo número elevado de falsos negativos. Dessa forma, mesmo com o uso de técnicas de balanceamento como o SMOTE durante o treinamento, a acurácia, isoladamente, não é suficiente para avaliar o desempenho do modelo, sendo necessário considerar métricas complementares.
</p>

---

<ul>
    <li>
        <h3><strong>Precision</strong></h3>
    </li>
</ul>

<p>
    Precision, ou em PT-BR, precisão, é uma métrica que mede com que frequência um modelo de machine learning prevê corretamente a classe positiva. Essa métrica pode ser obitida dividindo o total de previsões corretas da classe positiva (true positive) pelo total de casos que o modelo previu como positivas (true positive, false positives).
</p>

<p>
    <b>Definição básica:</b>
</p>

$$
\text{Precisão} =
\frac{\text{verdadeiros positivos}}
{\text{verdadeiros positivos} + \text{falsos positivos}}
$$

<p>
    <b>Baseada na matriz de confusão:</b>
</p>

$$
\text{Precisão} = \frac{TP}{TP + FP}
$$

<p>
    No <code>scikit-learn</code>, a função <code>precision_score()</code> é utilizada para calcular a precisão do modelo. O valor retornado está na escala de <code>0</code> a <code>1</code>, onde <code>1</code> corresponde a <code>100%</code> de precisão.
</p>

In [42]:
precision = metrics.precision_score(y_test, y_pred, pos_label=1)
print("Pecision: " + str(precision))

Pecision: 0.55893536121673


<p>
    Como pode ser observado, o modelo apresenta uma precisão de 0,55 (ou 55%). Isso significa que, entre todas as ocorrências classificadas como inadimplentes, aproximadamente dois terços realmente pertenciam a essa categoria. Embora esse resultado indique que as previsões positivas possuem uma confiabilidade razoável, ainda existe uma quantidade relevante de clientes adimplentes classificados incorretamente como inadimplentes, caracterizando a presença de falsos positivos.
</p>

---

<ul>
    <li>
        <h3><strong>Recall</strong></h3>
    </li>
</ul>

<p>
    Recall é uma métrica que mede a frequência com que o modelo de machine learning identifica corretamente a classe inadimplente (true positives) dentre todas as amostras inadimplentes reais no conjunto de dados. Essa métrica é calculada dividindo o total de verdadeiros positivos pelo total de inadimplentes reais, esse total é obtido pela soma de verdadeiros positivos e falsos negativos.
</p>

<p>
    <b>Definição básica:</b>
</p>

$$
\text{Recall} =
\frac{\text{verdadeiros positivos}}
{\text{verdadeiros positivos} + \text{falsos negativos}}
$$

<p>
    <b>Baseada na matriz de confusão:</b>
</p>

$$
\text{Recall} = \frac{TP}{TP + FN}
$$

<p>
    No <code>scikit-learn</code>, a função <code>recall_score()</code> é utilizada para calcular a precisão do modelo. O valor retornado está na escala de <code>0</code> a <code>1</code>, onde <code>1</code> corresponde a <code>100%</code> de precisão.
</p>

In [43]:
recall = metrics.recall_score(y_test, y_pred)
print("Recall: " + str(recall))

Recall: 0.4431047475508666


<p>
    O modelo apresentou um Recall de 0,44 (ou 44%) no conjunto de teste, o que significa que ele consegue identificar pouco mais de um terço dos clientes que realmente se tornarão inadimplentes. No contexto de risco de crédito, esse resultado é preocupante, pois indica que cerca de 63% dos casos de inadimplência não são detectados pelo modelo, sendo classificados incorretamente como clientes adimplentes (falsos negativos). Como consequência, uma parcela significativa dos clientes de maior risco pode passar despercebida, gerando potenciais prejuízos financeiros para a instituição.
</p>

---

<ul>
    <li>
        <h3><strong>F1-Score</strong></h3>
    </li>
</ul>

<p>
    F1-Score é uma métrica que mede o equilíbrio entre precision e recall de um modelo de machine learning, sendo especialmente útil quando há desbalanceamento entre as classes. Essa métrica é calculada por meio da média harmônica entre precisão e recall, penalizando situações em que uma dessas métricas apresenta valor baixo.
</p>

<p>
    <b>Definição básica:</b>
</p>

$$
\text{F1-Score} =
\frac{2 \times \text{Precisão} \times \text{Recall}}
{\text{Precisão} + \text{Recall}}
$$

In [44]:
f_score = metrics.f1_score(y_test, y_pred)
print("F1-Score: " + str(f_score))

F1-Score: 0.4943253467843632


<p>
    O modelo apresentou um F1-Score de 0,49 para a classe de inadimplência. Essa métrica combina precisão e recall em uma única medida, fornecendo uma visão mais equilibrada do desempenho do modelo. O valor obtido reflete que, apesar da precisão ser moderadamente satisfatória, o baixo recall impacta negativamente o resultado final. Dessa forma, o F1-Score evidencia que o modelo ainda possui dificuldades para identificar uma parcela significativa dos clientes inadimplentes, indicando oportunidades de melhoria no processo de classificação.
</p>

<h2>
    Conclusão
</h2>

<p>
    De modo geral, os resultados obtidos demonstram que o modelo possui um desempenho satisfatório na classificação da classe majoritária, apresentando uma acurácia de 79% e uma precisão de 55% para a identificação de clientes inadimplentes. No entanto, o baixo valor de recall (44%) evidencia que uma parcela significativa dos clientes que realmente se tornarão inadimplentes não é identificada pelo modelo. O F1-Score de 0,49 reforça essa visão, indicando um desequilíbrio entre a capacidade de realizar previsões corretas e a de recuperar efetivamente os casos positivos. Embora a aplicação do SMOTE durante o treinamento tenha contribuído para reduzir os efeitos do desbalanceamento das classes, os resultados sugerem que ainda há espaço para melhorias, especialmente na detecção da inadimplência. Dessa forma, o modelo apresenta potencial para auxiliar na análise de risco de crédito, mas requer ajustes adicionais para aumentar sua capacidade de identificar clientes inadimplentes de forma mais eficiente.
</p>